In [1]:
from IPython.lib.deepreload import original_reload


In [85]:
#!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126
#!pip install opencv-python
#!pip install "transformers[torch]"
#!pip install scipy

  Obtaining dependency information for scipy from https://files.pythonhosted.org/packages/a2/84/dc08d77fbf3d87d3ee27f6a0c6dcce1de5829a64f2eae85a0ecc1f0daa73/scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------------ ------------------------- 20.5/61.0 kB 330.3 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 650.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/36.5 MB 9.3 MB/s eta 0:00:04
   - -------------------------------------- 1.2/36.5 MB 15.1 MB/s eta 0:00:03
   -- ------------------------------------- 2.0/36.5 MB 16.2 MB/s eta 0:00:03
   --- ------------------------------------ 2.9/36.5 MB 17.0 MB/s eta 0:00:02
   ---- ----------------------------------- 3.8/36.5 MB 17.4 MB/s eta 0:00:02
   ----- ---------------------------------- 4.7/36.5 MB 17.7 MB/s eta 0:00:02
   ------ ----------------


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import numpy as np
import torch
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation, SamModel, SamProcessor, pipeline
from PIL import Image
from scipy.ndimage import distance_transform_edt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

D:\Programming Projects\TShirtMaking\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
mask_generator = pipeline(
    "mask-generation",
    model="facebook/sam-vit-large",
    device=device  # use -1 for CPU
)
backup_mask_generator = pipeline(
    "image-segmentation",
    model="shi-labs/oneformer_ade20k_swin_large",
    device=device  # use -1 for CPU
)

Loading weights: 100%|██████████| 482/482 [00:00<00:00, 4493.18it/s]
The image processor of type `SamImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 826/826 [00:00<00:00, 28075.32it/s]


In [28]:
original_image = cv2.imread("PrebendsEdit.jpg")
#original_image = cv2.imread("Bluebell.JPG")
#original_image = cv2.imread("Cas.jpg")

In [29]:
cv2.imwrite("debug.png", original_image)

True

In [30]:
scales = [original_image]
for i in range(4):
    scales.append(cv2.resize(scales[-1], None, fx=0.5, fy=0.5, interpolation=cv2.INTER_CUBIC))
cv2.imwrite("debug.png", scales[-1])


True

In [31]:
def get_overlay(image_to_segment):
    image_rgb = cv2.cvtColor(image_to_segment, cv2.COLOR_BGR2RGB)

    pil_image = Image.fromarray(image_rgb)

    # Run mask generation
    outputs = mask_generator(pil_image)

    # outputs is a list of dicts:
    # [{'mask': np.array(H,W), 'score': float}, ...]

    overlay = image_to_segment.copy()
    segments = np.zeros_like(overlay)
    np.random.seed(42)

    masks = outputs["masks"]



    # masks: list of boolean masks (H, W)
    H, W = masks[0].shape

    img_size = H * W

    masks_sorted = sorted(masks, key=lambda m: -m.sum())


    for i in range(len(masks_sorted) - 1):
        for j in range(i + 1, len(masks_sorted)):
            masks_sorted[j][masks_sorted[i]] = False


    masks_sorted = [x for x in masks_sorted if x.sum() > img_size * 0.05]

    label_map = torch.zeros((H, W), dtype=torch.int32)

    for idx, mask in enumerate(masks_sorted, 1):
        label_map[mask] = idx

    if (label_map == 0).sum() > img_size * 0.4:
        backup_outputs = backup_mask_generator(pil_image)
        backup_masks = [np.array(x["mask"]) > 0 for x in backup_outputs]

        masks_sorted = masks_sorted + backup_masks

        for i in range(len(masks_sorted) - 1):
            for j in range(i + 1, len(masks_sorted)):
                masks_sorted[j][masks_sorted[i]] = False


        masks_sorted = [x for x in masks_sorted if x.sum() > img_size * 0.05]

        label_map = torch.zeros((H, W), dtype=torch.int32)

        for idx, mask in enumerate(masks_sorted, 1):
            label_map[mask] = idx



    mask_union = label_map > 0




    dist, indices = distance_transform_edt(~mask_union, return_indices=True)

    label_map[~mask_union] = label_map[
        indices[0][~mask_union],
        indices[1][~mask_union]
    ]

    mask_ids = np.unique(label_map)
    masks = [(label_map == mid) for mid in mask_ids]


    for mask in masks:
        mask = np.array(mask)  # ensure numpy
        color = np.random.randint(0, 255, size=3)

        overlay[mask] = (
                0.6 * overlay[mask] +
                0.4 * color
        ).astype(np.uint8)

        segments[mask] = color



    cv2.imwrite("SegmentationOvelay.png", overlay)
    cv2.imwrite("Segmentation.png", segments)


    return masks

In [32]:
segments = get_overlay(scales[1])

C:\Users\Enego\AppData\Local\Temp\ipykernel_7648\3842908048.py:77: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  mask = np.array(mask)  # ensure numpy


In [30]:
len(t[t>1])

1444069

In [37]:
t

torch.return_types.max(
values=tensor([[29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29],
        ...,
        [29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29]]),
indices=tensor([[29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29],
        ...,
        [29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29],
        [29, 29, 29,  ..., 29, 29, 29]]))

In [33]:
len(t)

1170

In [34]:
t.shape

torch.Size([1170, 2080])

In [35]:
1170 * 2080

2433600

In [58]:
def BlurImages(img):
    #return cv2.bilateralFilter(img, 20, 100, 100)
    blur = cv2.GaussianBlur(img, (21, 21), 4)
    cv2.imwrite("debugBlur.png", blur)
    return blur

def MakeOdd(num):
    if num <= 1:
        print("Kernal Size was too small so set to 3")
        return 3
    return num - num%2 + 1

In [68]:
def FindEdges(sizes, size, pixel_detail_factor, minimum_edge_threshold, diff_from_median_threshold):
    img = sizes[size]
    blurred = BlurImages(img)

    gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)

    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)  # Horizontal edges
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)  # Vertical edges

    # Compute gradient magnitude
    gradient_magnitude = cv2.magnitude(sobelx, sobely)

    # Convert to uint8
    gradient_magnitude = cv2.convertScaleAbs(gradient_magnitude)


    edges = gradient_magnitude
    cv2.imwrite("debugEdges.png", edges)



    adujsted_threshold = minimum_edge_threshold * 255




    median_edges = cv2.medianBlur(edges, MakeOdd(128 // (2**size) + 1))

    colour_mask = np.bitwise_and((edges > median_edges * diff_from_median_threshold), (edges > adujsted_threshold)) * 255

    cv2.imwrite("debugMask.png", colour_mask)

    blurred_edges = cv2.GaussianBlur(edges, (15, 15), 0)


    alt_median_edges = cv2.medianBlur(blurred_edges, 128 // (2**size) + 1)

    alt_colour_mask = np.bitwise_and((blurred_edges > alt_median_edges * diff_from_median_threshold), (blurred_edges > adujsted_threshold)) * 255

    cv2.imwrite("altDebugMask.png", alt_colour_mask)

    cv2.imwrite("alt.png", np.append(img, alt_colour_mask[:, :, np.newaxis], axis=2))
    #out = np.append(img, colour_mask[:, :, np.newaxis], axis=2)

    return alt_colour_mask > 0

In [69]:
edges = FindEdges(scales, size=1,
                  pixel_detail_factor = 1, # Use this to adjust the relative information per pixel.
                  # debugEdges.JPG
                  #
                  # Edge Thresholding:
                  minimum_edge_threshold = 0.05,
                  diff_from_median_threshold = 1.5
                  # debugMask
                  )

error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'imwrite'
> Overload resolution failed:
>  - img data type = bool is not supported
>  - Expected Ptr<cv::UMat> for argument 'img'


In [ ]:
colours = edges *

In [67]:
edges

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]],
      shape=(1170, 2080))

In [103]:
def ImageComponentAnalysis(sizes, size):
    img = sizes[size]
    blurred = cv2.medianBlur(img, 7)



    bin_size = 48


    binned_image = blurred - (blurred)%bin_size# + bin_size//2

    cv2.imwrite("ICA.png", binned_image)




In [104]:
ImageComponentAnalysis(scales, 1)

Loading weights: 100%|██████████| 482/482 [00:00<00:00, 889.41it/s, Materializing param=vision_encoder.pos_embed]                                                  


C:\Users\Enego\AppData\Local\Temp\ipykernel_13892\3567239367.py:18: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  mask = np.array(mask)  # ensure numpy


True